## `host` & `none` modes

Two drivers sit at the opposite extremes of "how much network does this container get": **all of the host's**, or **none at all.**

**`--network host` — share the host's stack.** The container gets **no namespace of its own**; it sees and binds the host's interfaces directly.

```bash
docker run --network host nginx     # nginx now listens on the HOST's port 80
```

- **You lose:** port mapping (`-p` is ignored — you bind host ports directly), isolation (two host-mode containers can't both take port 80), and the bridge (no `docker0`, no embedded DNS).
- **You gain:** no NAT overhead (useful for very-high-throughput servers, load balancers, packet inspection), and raw access to the host's network config — `netstat`, `tcpdump`, `iptables` inside the container see exactly what the host sees.
- **Trade-off:** it breaks the "container is isolated" promise for networking. Use it sparingly, with a specific reason. And note: on **Docker Desktop (Mac/Windows)** it's only partially supported — it shares the *VM's* network, not your laptop's, so don't rely on it cross-platform.

**`--network none` — no network.** The namespace contains only loopback `lo`: no `eth0`, no DNS, no outbound.

```bash
docker run --rm --network none alpine ip addr
```

Use it for sandboxed batch jobs that should never phone home, or forensic analysis of an untrusted image. It isn't necessarily permanent — you can `docker network connect` a network later to add an interface.

Between these extremes sits `bridge`, the default, where the rest of the module lives: private by default, and traffic comes *in* only when you publish a port — which is next.